# 07 - Full evaluation report and production configuration (Day 13)

**Question:** with every detector judged by the *same* harness on a *final held-out* test day, which one - or which combination - should run in production, and what does each family cost and buy?

What this notebook does (all metrics come from `src/models/evaluator.py`):

1. Re-creates the Day 8-12 detectors with the notebook 06 protocol (base training on **A**, meta-learner on **B**).
2. Simulates a **brand-new test day D** (new seed, new day) that no earlier notebook has seen.
3. Picks every model's **operating threshold on B only** (threshold sweep), never on D. D is scored once.
4. Builds the leaderboard (weighted F1, point-adjusted F1, AUC-PR, FPR, MTTD, latency), exports it, and writes one HTML report per model to `reports/`.
5. Discusses the statistical-vs-LSTM trade-offs and selects the production configuration under explicit constraints.

**Reading the metrics.** *F1* is row-level. *PA F1* (point-adjusted) credits a whole anomaly event if any row in it is flagged - standard in the TSAD literature but generous, so it is always shown next to *Event recall* and plain F1. *MTTD* is the mean time from event start to first flag over the events that were detected - in scored steps (one step = one scored event, every 10th row per sensor) and in seconds; missed events are not in the mean, so read it with *Event recall*.

In [1]:
import sys, os, time, logging, warnings
from pathlib import Path
from datetime import datetime, UTC

import numpy as np
import pandas as pd
import plotly.io as pio
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
logging.getLogger("src").setLevel(logging.WARNING)
warnings.filterwarnings("ignore")
pio.renderers.default = "notebook_connected"

from IPython.display import HTML, Markdown, display
from src.models.offline_dataset import simulate_labeled_features
from src.models.feature_engineering import OnlineScaler
from src.models.lstm_autoencoder import make_sequences, select_features, train_detector
from src.models.lstm_classifier import DatasetBuilder, train_classifier
from src.models.isolation_forest_detector import IsolationForestDetector
from src.models.statistical_detector import UnifiedStatisticalDetector
from src.models.ensemble_detector import MEMBERS, EnsembleDetector, collect_member_scores
from src.models.evaluator import (
    AnomalyDetectionEvaluator, DetectionResults, EvaluationReporter, ModelLeaderboard,
    entity_heatmap_figure, leaderboard_figure, pr_figure, roc_figure, sweep_figure,
)
from src.utils.config import LSTMClassifierConfig, get_config

sns.set_theme(style="whitegrid", context="notebook", palette="colorblind")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold"})
cfg = get_config()
SEED, EPOCHS = 7, 15
BASE_H, SEG_H = 10, 6          # simulated hours: base training / meta-learner B / test D
N_THRESHOLDS = 100
REPORT_DIR = ROOT / "reports"
LABELS = {"statistical": "Statistical", "isolation_forest": "Isolation Forest", "lstm_autoencoder": "LSTM-AE",
          "lstm_classifier": "LSTM-Cls", "weighted": "Weighted avg", "stacked": "Stacked (LightGBM)"}

## 1. Data - three independent simulations

Same generator and features as notebook 06 (production path: simulator -> windowed aggregator -> feature engineer). **D is new**: seed 17 on 23 Nov, so it was never used to choose a model, a threshold or a hyper-parameter.

In [2]:
D = lambda day, h: datetime(2026, 11, day, h, 0, tzinfo=UTC)
t0 = time.time()
sims = {
    "A: base training": simulate_labeled_features(BASE_H * 3600, start_time=D(20, 19), seed=7)["warehouse"],
    "B: meta-learner + thresholds": simulate_labeled_features(SEG_H * 3600, start_time=D(21, 22), seed=11)["warehouse"],
    "D: FINAL TEST (held out)": simulate_labeled_features(SEG_H * 3600, start_time=D(23, 22), seed=17)["warehouse"],
}
base, meta, test = sims.values()
rows = []
for name, ds in sims.items():
    c = ds.type_counts()
    rows.append({"simulation": name, "events": len(ds), "anomaly rate": ds.anomaly_rate,
                 **{k: c.get(k, 0) for k in ("point", "contextual", "collective")}})
display(pd.DataFrame(rows).set_index("simulation").style.format({"events": "{:,}", "anomaly rate": "{:.2%}"}))
print(f"simulated in {time.time() - t0:.0f}s; {len(base.feature_names)} features, {len(set(base.entity_ids))} sensors")

,events,anomaly rate,point,contextual,collective
simulation,,,,,
A: base training,"432,000",4.78%,8334,10085,2214
B: meta-learner + thresholds,"259,200",5.47%,4800,8192,1199
D: FINAL TEST (held out),"259,200",5.43%,4985,8024,1070


simulated in 75s; 30 features, 12 sensors


## 2. Train the four base detectors on A only

Identical to notebook 06: statistical (online, no training), Isolation Forest (contamination 3%), LSTM autoencoder (normal windows only, clock features excluded), LSTM classifier (class weights + SMOTE). One scaler fitted on the warm normal rows of A.

In [3]:
names = tuple(base.feature_names)
scaler = OnlineScaler(len(names), clip=10.0)
scaler.fit_update(base.subset(base.warm & ~base.y).X)

t0 = time.time()
forest = IsolationForestDetector(names, source_type="warehouse", contamination=0.03, n_estimators=200,
                                 random_state=SEED).fit(base.X[base.warm][::5])
print(f"Isolation Forest fitted ({time.time() - t0:.0f}s)")

ae_names, cols = select_features(names)
W = make_sequences(base, scaler, seq_len=cfg.models.lstm_autoencoder.seq_len, stride=10, columns=cols)
Wn = W.subset(W.normal); cut = np.quantile(Wn.end_timestamps, 0.8)
trn, val = Wn.subset(Wn.end_timestamps < cut), Wn.subset(Wn.end_timestamps >= cut)
t0 = time.time()
ae, ae_hist = train_detector(trn.x, val.x, ae_names, cfg.models.lstm_autoencoder.model_copy(update={"max_epochs": EPOCHS}),
                             columns=cols, log_dir=None)
ae.mc_passes = 3
print(f"LSTM autoencoder: {ae_hist.epochs_run} epochs in {time.time() - t0:.0f}s")

data = DatasetBuilder(stride=10, smote_ratio=0.3, seed=SEED).build(base, scaler)
t0 = time.time()
clf, clf_hist = train_classifier(data, LSTMClassifierConfig(max_epochs=EPOCHS, early_stopping_patience=8, batch_size=128),
                                 log_dir=None, num_workers=0, seed=SEED)
print(f"LSTM classifier: {clf_hist.epochs_run} epochs in {time.time() - t0:.0f}s (val macro-F1 {clf_hist.best_score:.3f})")

Isolation Forest fitted (1s)


LSTM autoencoder: 15 epochs in 279s


LSTM classifier: 15 epochs in 187s (val macro-F1 0.839)


## 3. Score tables for B and D, and the ensemble

`collect_member_scores` scores one event every 10 rows per sensor with all four detectors. The meta-learner is fitted on **B** (5-fold CV grouped by sensor), exactly as in notebook 06. A weighted-average ensemble (0.2/0.3/0.25/0.25, no stacker) is evaluated too.

In [4]:
def table(ds):
    return collect_member_scores(ds, scaler, statistical=UnifiedStatisticalDetector.production(), isolation_forest=forest,
                                 lstm_autoencoder=ae, lstm_classifier=clf, stride=10)
t0 = time.time()
tab_b, tab_d = table(meta), table(test)
print(f"scored in {time.time() - t0:.0f}s: B {len(tab_b):,} events ({int(tab_b.y.sum())} anomalous), "
      f"D {len(tab_d):,} events ({int(tab_d.y.sum())} anomalous)")

ens = EnsembleDetector.from_config(cfg.models.ensemble)
weighted = EnsembleDetector.from_config(cfg.models.ensemble)
rep = ens.fit_meta_learner(tab_b)
print(f"meta-learner out-of-fold on B: F1 {rep.f1:.3f}, AP {rep.average_precision:.3f}, threshold {rep.threshold:.3f}")

def all_scores(tab):
    s = {m: tab.scores[:, i] for i, m in enumerate(MEMBERS)}
    s["weighted"] = weighted.score_table(tab)
    s["stacked"] = ens.score_table(tab)
    return s
scores_b, scores_d = all_scores(tab_b), all_scores(tab_d)
coverage = pd.Series({LABELS[k]: float(np.isfinite(v).mean()) for k, v in scores_d.items()}, name="share of D events scored")
display(coverage.to_frame().style.format("{:.1%}"))

scored in 74s: B 24,888 events (1456 anomalous), D 24,888 events (1384 anomalous)


meta-learner out-of-fold on B: F1 0.746, AP 0.796, threshold 0.347


,share of D events scored
Statistical,100.0%
Isolation Forest,100.0%
LSTM-AE,100.0%
LSTM-Cls,100.0%
Weighted avg,100.0%
Stacked (LightGBM),100.0%


## 4. Operating thresholds - chosen on B, applied to D

Every detector reports `0.5` as its own decision boundary, but that is rarely the F1-optimal point. `sweep_thresholds` finds the optimum on **B**; that threshold is then frozen and applied to D. The stacked ensemble was *fitted* on B, so a sweep on B would be optimistic - its probability is already mapped so the out-of-fold optimum sits at 0.5, and it keeps 0.5.

In [5]:
ev = AnomalyDetectionEvaluator()
thr = {}
rows = []
for k, v in scores_b.items():
    sw = ev.sweep_thresholds(tab_b.y, v, N_THRESHOLDS, entity_ids=tab_b.entity_ids, timestamps=tab_b.timestamps)
    thr[k] = 0.5 if k == "stacked" else (sw.optimal_threshold if np.isfinite(sw.optimal_threshold) else 0.5)
    rows.append({"model": LABELS[k], "default": 0.5, "B-optimal": sw.optimal_threshold, "used": thr[k], "F1 on B at used": sw.at(thr[k])["f1"]})
display(pd.DataFrame(rows).set_index("model").style.format("{:.3f}"))

,default,B-optimal,used,F1 on B at used
model,,,,
Statistical,0.500,0.505,0.505,0.553
Isolation Forest,0.500,0.646,0.646,0.473
LSTM-AE,0.500,0.434,0.434,0.157
LSTM-Cls,0.500,0.939,0.939,0.675
Weighted avg,0.500,0.465,0.465,0.680
Stacked (LightGBM),0.500,0.485,0.500,0.765


## 5. Final held-out evaluation on D

Two views of the same scores: **default boundary (0.5)** - what each detector does out of the box - and **B-tuned thresholds** - what a deployment would run. Latency is the measured batch scoring time per event on D (single CPU, features already computed).

In [6]:
def timed(fn, n):
    fn(); t0 = time.perf_counter(); fn(); return (time.perf_counter() - t0) / n * 1000

# --- latency per event (ms) --------------------------------------------------
n_d = len(tab_d)
lat = {}
t0 = time.perf_counter()
stat = UnifiedStatisticalDetector.production(); col = next(c for c in names if c.endswith("__last")); ci = names.index(col)
for i in range(len(test)):
    stat.detect(str(test.entity_ids[i]), float(test.X[i, ci]), float(test.timestamps[i]), None if test.zones[i] is None else str(test.zones[i]))
lat["statistical"] = (time.perf_counter() - t0) / len(test) * 1000          # per streamed row (it sees every row)
rows_idx = np.arange(0, len(test), 10)[:n_d]
lat["isolation_forest"] = timed(lambda: forest.score_batch(test.X[rows_idx]), len(rows_idx))
w_ae = make_sequences(test, scaler, seq_len=ae.seq_len, stride=10, columns=ae.columns)
lat["lstm_autoencoder"] = timed(lambda: ae.score_sequences(w_ae.x), len(w_ae))
w_cl = make_sequences(test, scaler, seq_len=clf.seq_len, stride=10, columns=clf.columns)
lat["lstm_classifier"] = timed(lambda: clf.classify_sequences(w_cl.x), len(w_cl))
lat["weighted"] = sum(lat[m] for m in MEMBERS)   # all four members + a weighted mean
lat["stacked"] = lat["weighted"] + timed(lambda: ens.score_table(tab_d), n_d)   # four members + LightGBM stacker
display(pd.Series({LABELS[k]: v for k, v in lat.items()}, name="ms / event").to_frame().style.format("{:.3f}"))

# --- evaluation ----------------------------------------------------------------
board_default = ev.evaluate_scorers(tab_d, {LABELS[k]: v for k, v in scores_d.items()},
                                    latency_ms={LABELS[k]: lat[k] for k in scores_d})
board = ev.evaluate_scorers(tab_d, {LABELS[k]: v for k, v in scores_d.items()},
                            thresholds={LABELS[k]: thr[k] for k in scores_d},
                            latency_ms={LABELS[k]: lat[k] for k in scores_d}, n_thresholds=N_THRESHOLDS)

,ms / event
Statistical,0.028
Isolation Forest,0.008
LSTM-AE,0.894
LSTM-Cls,0.083
Weighted avg,1.013
Stacked (LightGBM),1.017


### 5.1 Leaderboard - B-tuned thresholds (the deployable view)

In [7]:
table_tuned = board.generate_comparison_table()
fmt = {c: "{:.3f}" for c in table_tuned.columns if c != "model"} | {"mttd": "{:.1f}", "latency_ms": "{:.3f}", "fpr": "{:.4f}"}
display(table_tuned.style.format(fmt, na_rep="-")
        .highlight_max(subset=["weighted_f1", "f1", "pa_f1", "auc_pr", "roc_auc", "event_recall"], color="#c7e9c0")
        .highlight_min(subset=["fpr", "fnr", "mttd", "latency_ms"], color="#c7e9c0"))

,model,weighted_f1,macro_f1,f1,precision,recall,pa_f1,auc_pr,roc_auc,fpr,fnr,mttd,mttd_s,event_recall,threshold,oracle_f1,oracle_threshold,latency_ms
rank,,,,,,,,,,,,,,,,,,
1,Stacked (LightGBM),0.551,0.515,0.644,0.930,0.492,0.664,0.736,0.928,0.0022,0.508,0.0,0.358,0.517,0.500,0.698,0.202,1.017
2,LSTM-Cls,0.454,0.454,0.576,0.947,0.414,0.593,0.574,0.826,0.0014,0.586,0.0,0.372,0.436,0.939,0.579,0.909,0.083
3,Weighted avg,0.421,0.406,0.562,0.787,0.436,0.582,0.542,0.843,0.0069,0.564,0.0,0.471,0.459,0.465,0.565,0.485,1.013
4,Statistical,0.398,0.420,0.541,0.998,0.371,0.554,0.540,0.891,0.0000,0.629,0.0,0.256,0.391,0.505,0.541,0.505,0.028
5,Isolation Forest,0.301,0.301,0.459,0.745,0.332,0.469,0.471,0.829,0.0067,0.668,0.0,0.242,0.351,0.646,0.459,0.646,0.008
6,LSTM-AE,0.072,0.056,0.150,0.085,0.629,0.153,0.078,0.647,0.3993,0.371,0.0,0.168,0.644,0.434,0.150,0.434,0.894


### 5.2 Leaderboard - default boundary (0.5), for comparison

In [8]:
t_def = board_default.generate_comparison_table()
display(t_def[["model", "weighted_f1", "f1", "precision", "recall", "pa_f1", "auc_pr", "roc_auc", "fpr", "mttd", "event_recall"]]
        .style.format({c: "{:.3f}" for c in ["weighted_f1", "f1", "precision", "recall", "pa_f1", "auc_pr", "roc_auc", "event_recall"]}
                      | {"fpr": "{:.4f}", "mttd": "{:.1f}"}, na_rep="-"))

,model,weighted_f1,f1,precision,recall,pa_f1,auc_pr,roc_auc,fpr,mttd,event_recall
rank,,,,,,,,,,,
1,Stacked (LightGBM),0.551,0.644,0.930,0.492,0.664,0.736,0.928,0.0022,0.0,0.517
2,Weighted avg,0.423,0.560,0.843,0.419,0.576,0.542,0.843,0.0046,0.0,0.440
3,LSTM-Cls,0.418,0.559,0.795,0.431,0.579,0.574,0.826,0.0066,0.0,0.453
4,Isolation Forest,0.288,0.457,0.648,0.353,0.468,0.471,0.829,0.0113,0.0,0.372
5,Statistical,0.210,0.373,0.305,0.478,0.389,0.540,0.891,0.0642,0.1,0.497
6,LSTM-AE,0.068,0.142,0.084,0.470,0.144,0.078,0.647,0.3030,0.0,0.480


In [9]:
fig = leaderboard_figure(table_tuned, metrics=("f1", "pa_f1", "auc_pr"))
fig.update_layout(title="Final test D - B-tuned thresholds")
fig.show()

### 5.3 Which family wins on which anomaly type?

Recall per type at the tuned thresholds, and F1 per type. Collective anomalies are the ones a per-point z-score cannot see.

In [10]:
display(Markdown("**Recall by anomaly type**"))
display(board.per_type_table("recall").style.format("{:.1%}", na_rep="-").background_gradient(cmap="Blues", vmin=0, vmax=1))
display(Markdown("**F1 by anomaly type**"))
display(board.per_type_table("f1").style.format("{:.3f}", na_rep="-").background_gradient(cmap="Blues", vmin=0, vmax=1))
ev_counts = board.report("Stacked (LightGBM)")
display(Markdown("Events in D: " + ", ".join(f"{t.type} {t.events}" for t in ev_counts.per_type.values()) +
                 f" | rows: " + ", ".join(f"{t.type} {t.support}" for t in ev_counts.per_type.values())))

**Recall by anomaly type**

,collective,contextual,point
Stacked (LightGBM),20.0%,23.3%,100.0%
LSTM-Cls,15.2%,10.7%,99.8%
Weighted avg,22.9%,13.5%,99.8%
Statistical,10.5%,3.8%,100.0%
Isolation Forest,6.7%,1.7%,92.8%
LSTM-AE,49.5%,47.5%,92.4%


**F1 by anomaly type**

,collective,contextual,point
Stacked (LightGBM),0.237,0.359,0.949
LSTM-Cls,0.209,0.186,0.966
Weighted avg,0.164,0.202,0.852
Statistical,0.188,0.074,0.999
Isolation Forest,0.052,0.029,0.821
LSTM-AE,0.011,0.072,0.085


Events in D: collective 56, contextual 776, point 465 | rows: collective 105, contextual 807, point 472

### 5.4 Threshold sweeps on D (diagnostic only)

These curves show how much headroom each model has. The dashed line is the threshold actually used (chosen on B); the dotted line is the *oracle* optimum on D, which no deployment can know. A large gap between the two means the score ranks well but the threshold generalises poorly.

In [11]:
for label in ("Isolation Forest", "LSTM-AE", "Stacked (LightGBM)"):
    fig = sweep_figure(board.sweep(label), board.report(label).threshold)
    fig.update_layout(title=f"Threshold sweep on D - {label}")
    fig.show()

### 5.5 ROC and precision-recall, all models

In [12]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import precision_recall_curve, roc_curve
fig = make_subplots(rows=1, cols=2, subplot_titles=("ROC", "Precision-recall"))
y = tab_d.y
for k, v in scores_d.items():
    s = np.nan_to_num(v, nan=0.0)
    f, t, _ = roc_curve(y, s); p, r, _ = precision_recall_curve(y, s)
    kw = dict(name=LABELS[k], legendgroup=k, line=dict(width=3.5 if k == "stacked" else 1.8))
    fig.add_trace(go.Scatter(x=f, y=t, mode="lines", **kw), row=1, col=1)
    fig.add_trace(go.Scatter(x=r, y=p, mode="lines", showlegend=False, **kw), row=1, col=2)
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color="grey", dash="dot"), showlegend=False), row=1, col=1)
fig.update_layout(template="plotly_white", height=430, legend=dict(orientation="h", y=-0.2))
fig.update_xaxes(title_text="FPR", row=1, col=1); fig.update_yaxes(title_text="TPR", row=1, col=1)
fig.update_xaxes(title_text="Recall", row=1, col=2); fig.update_yaxes(title_text="Precision", row=1, col=2)
fig.show()

### 5.6 Per-sensor breakdown of the stacked ensemble

In [13]:
res_stacked = DetectionResults.from_member_table(tab_d, scores_d["stacked"])
per_entity = ev.compute_per_entity_metrics(res_stacked, threshold=thr["stacked"])
display(per_entity.sort_values("f1").head(10).style.format({c: "{:.3f}" for c in per_entity.columns if c not in ("n", "n_anomalies", "n_flagged", "events", "events_detected")}, na_rep="-"))
entity_heatmap_figure(per_entity).show()

,n,n_anomalies,anomaly_rate,n_flagged,precision,recall,f1,fpr,roc_auc,auc_pr,pa_f1,mttd,events,events_detected
entity_id,,,,,,,,,,,,,,
wh2-A-temp-01,2074,114,0.055,48,0.958,0.404,0.568,0.001,0.943,0.751,0.568,0.000,111,46
wh1-C-temp-01,2074,109,0.053,45,1.000,0.413,0.584,0.000,0.902,0.710,0.594,0.000,100,45
wh1-A-temp-01,2074,113,0.054,59,0.881,0.460,0.605,0.004,0.914,0.680,0.621,0.041,105,49
wh1-B-temp-01,2074,105,0.051,49,0.959,0.448,0.610,0.001,0.921,0.722,0.619,0.000,96,47
wh2-C-temp-01,2074,119,0.057,58,0.948,0.462,0.621,0.002,0.940,0.765,0.637,0.038,112,53
wh1-A-hum-01,2074,115,0.055,61,0.902,0.478,0.625,0.003,0.919,0.731,0.670,0.093,102,54
wh2-A-hum-01,2074,121,0.058,69,0.913,0.521,0.663,0.003,0.910,0.704,0.691,0.065,113,62
wh2-B-hum-01,2074,130,0.063,69,0.957,0.508,0.663,0.002,0.908,0.715,0.715,0.109,118,64
wh2-B-temp-01,2074,109,0.053,56,0.982,0.505,0.667,0.001,0.949,0.784,0.667,0.000,107,55


## 6. Auto-generated reports and leaderboard export

One self-contained HTML per model (metrics table, confusion matrix, ROC/PR, threshold sweep, per-entity heatmap) under `reports/`, plus the leaderboard as CSV and HTML. Open any report in a browser and use *Print -> Save as PDF* for the PDF version.

In [14]:
reporter = EvaluationReporter(REPORT_DIR)
paths = {}
for k, v in scores_d.items():
    label = LABELS[k]
    res = DetectionResults.from_member_table(tab_d, v)
    pe = ev.compute_per_entity_metrics(res, threshold=thr[k])
    paths[label] = reporter.generate_report(label, board.report(label), y_true=tab_d.y, y_scores=v,
                                            sweep=board.sweep(label), per_entity=pe)
exported = board.export_leaderboard(REPORT_DIR / "leaderboard", title="Supply-chain anomaly detection - final test leaderboard")
exported_default = board_default.export_leaderboard(REPORT_DIR / "leaderboard_default_threshold", title="Leaderboard at default boundary 0.5")
for label, p in paths.items():
    print(f"{label:22s} {p.relative_to(ROOT)}")
print({k: str(v.relative_to(ROOT)) for k, v in exported.items()})

Statistical            reports/Statistical_20261123T211939Z.html
Isolation Forest       reports/Isolation_Forest_20261123T211940Z.html
LSTM-AE                reports/LSTM-AE_20261123T211940Z.html
LSTM-Cls               reports/LSTM-Cls_20261123T211940Z.html
Weighted avg           reports/Weighted_avg_20261123T211940Z.html
Stacked (LightGBM)     reports/Stacked_LightGBM_20261123T211940Z.html
{'csv': 'reports/leaderboard.csv', 'html': 'reports/leaderboard.html'}


## 7. Trade-offs: statistical methods versus LSTM

The table is computed from the results above; the reading follows it.

In [15]:
t = table_tuned.set_index("model")
stat_row, ifo, ae_row, cl_row, st = (t.loc[LABELS[k]] for k in ("statistical", "isolation_forest", "lstm_autoencoder", "lstm_classifier", "stacked"))
cmp = pd.DataFrame({
    "F1": t["f1"], "PA F1": t["pa_f1"], "Event recall": t["event_recall"], "FPR": t["fpr"],
    "MTTD (rows)": t["mttd"], "ms / event": t["latency_ms"],
})
rec = board.per_type_table("recall")
cmp = cmp.join(rec.add_prefix("recall: "))
display(cmp.style.format({c: "{:.3f}" for c in cmp.columns} | {"FPR": "{:.4f}", "MTTD (rows)": "{:.1f}", "ms / event": "{:.3f}"}, na_rep="-"))

,F1,PA F1,Event recall,FPR,MTTD (rows),ms / event,recall: collective,recall: contextual,recall: point
model,,,,,,,,,
Stacked (LightGBM),0.644,0.664,0.517,0.0022,0.0,1.017,0.200,0.233,1.000
LSTM-Cls,0.576,0.593,0.436,0.0014,0.0,0.083,0.152,0.107,0.998
Weighted avg,0.562,0.582,0.459,0.0069,0.0,1.013,0.229,0.135,0.998
Statistical,0.541,0.554,0.391,0.0000,0.0,0.028,0.105,0.038,1.000
Isolation Forest,0.459,0.469,0.351,0.0067,0.0,0.008,0.067,0.017,0.928
LSTM-AE,0.150,0.153,0.644,0.3993,0.0,0.894,0.495,0.475,0.924



### Reading the results (final test day D: 1,384 anomalous rows, 1,297 events)

**Statistical methods: fast, interpretable, and already enough for point anomalies.**
The statistical detector reaches point-anomaly recall 100% and F1 0.999 with zero false positives at the B-tuned threshold, at 0.03 ms per reading, and every alert can be explained by a z-score, an IQR fence or a CUSUM drift. There is no model to train, retrain or version. What it cannot do is see context: contextual recall is 3.8% and collective recall 10.5%, so overall event recall is only 39%. At its default boundary it is also the noisiest useful model (FPR 6.4%); the B-tuned threshold (0.505) is what makes it precise.

**LSTM models: the hypothesis is only partly supported here.**
* *LSTM autoencoder.* It has the highest recall on the hard types (collective 49.5%, contextual 47.5%), so it does see structure the statistical methods miss. But its ranking quality is poor (AUC-PR 0.078, AUC-ROC 0.65) and the threshold that maximises F1 on B still yields a 40% false-positive rate. Alone it is unusable; as an input to the stacker it adds signal.
* *LSTM classifier.* Collective recall 15.2% versus 10.5% for the statistical detector, and contextual recall 10.7% versus 3.8%. That is better, but modest, and the collective sample is small (56 events, 105 rows), so the gap is within noise. Its real strengths are precision (0.947) and the lowest false-alarm rate of any non-statistical model, at 0.08 ms per event.
* *Isolation Forest.* The cheapest model (0.008 ms) and a reasonable point detector (recall 92.8%) but the weakest on contextual and collective anomalies (1.7% and 6.7%).

**Why the stacked ensemble wins.** It is best on weighted F1 (0.551), F1 (0.644), PA F1, AUC-PR (0.736) and AUC-ROC (0.928), with precision 0.930 and FPR 0.22%. It has the best F1 on collective (0.237) and contextual (0.359) anomalies; on point anomalies it matches the best recall (100%) but the statistical detector (F1 0.999) and the LSTM classifier (0.966) are more precise. Only the LSTM autoencoder has higher event recall (64% vs 52%), and it gets there with a 40% false-alarm rate. A plain weighted average of the same four scores is clearly worse (F1 0.562, FPR 0.69%), so the gain comes from the learned combination, not from simply having four detectors.

**What to be careful about.**
* *Contextual anomalies dominate.* They are 776 of 1,297 events (60%) and every model is weak on them (best: 23.3%). Event recall for the best model is 51.7%, just above the 50% constraint. Improving contextual detection is the largest remaining lever, not further tuning of the point detectors.
* *Point-adjusted F1 adds little here.* Events are short (point anomalies are almost all single rows: 465 events for 472 rows), so PA F1 (0.664) is within 0.02 of plain F1 (0.644). It would matter on datasets with long events, which is why it stays in the suite but should not be read as a headline number.
* *MTTD is near zero (0.02-0.05 steps, 0.2-0.5 s)* because events of every type are short here (about 1-2 rows each: 465 events for 472 point rows, 776 for 807 contextual, 56 for 105 collective) and are caught at their first row, if at all. It would discriminate between models on longer events, but on this simulator it does not.
* *Single simulation, no confidence intervals.* One seed per segment, with a shortened training schedule (15 epochs, as in notebook 06). Differences of a few points between neighbouring models (for example LSTM-Cls versus the weighted average) should not be over-interpreted; the gap between the stacker and everything else is large enough to be credible.
* *Threshold choice matters.* Tuning on B instead of using 0.5 changes the ranking: the LSTM classifier moves from third (F1 0.559, FPR 0.7%) to second (F1 0.576, FPR 0.14%) and the statistical detector's FPR drops from 6.4% to 0.0%. The oracle thresholds on D (last columns of the leaderboard) show how much headroom remains; for the LSTM classifier it is small.


## 8. Production configuration

The leaderboard ranks by weighted F1, but production has constraints a ranking ignores. Candidate models must satisfy **all** of:

* false-positive rate <= 1% (an operator will not tolerate more alert noise),
* event recall >= 50% (at least half of the anomaly events are caught),
* latency <= 50 ms per event (the streaming budget on one CPU core).

Among the models that pass, the best-ranked one is selected. The chosen operating threshold, the metrics it achieved on the final test day and the severity thresholds from `config.alerts` are written to `reports/production_config.json`.

In [16]:
import json
CONSTRAINTS = dict(max_false_positive_rate=0.01, min_event_recall=0.5, max_latency_ms=50.0)
tt = table_tuned.set_index("model")
checks = pd.DataFrame({
    "FPR <= 1%": tt["fpr"] <= CONSTRAINTS["max_false_positive_rate"],
    "event recall >= 50%": tt["event_recall"] >= CONSTRAINTS["min_event_recall"],
    "latency <= 50 ms": tt["latency_ms"] <= CONSTRAINTS["max_latency_ms"],
})
checks["all"] = checks.all(axis=1)
display(checks.join(tt[["weighted_f1", "f1", "pa_f1"]]).style.format({"weighted_f1": "{:.3f}", "f1": "{:.3f}", "pa_f1": "{:.3f}"}))

chosen = board.recommend(**CONSTRAINTS)
relaxed = False
if chosen is None:                       # nothing meets every constraint: fall back to the best-ranked model and say so
    chosen, relaxed = board.best_model(), True
r = board.report(chosen)
display(Markdown(f"### Selected: **{chosen}** at threshold {r.threshold:.3f}" + ("  (no model met every constraint - best-ranked model used)" if relaxed else "")))
production = {
    "generated": datetime.now(UTC).isoformat(timespec="seconds"),
    "model": chosen,
    "constraints": CONSTRAINTS,
    "constraints_relaxed": relaxed,
    "operating_threshold": r.threshold,
    "threshold_chosen_on": "held-out segment B" if chosen != LABELS["stacked"] else "out-of-fold meta-learner CV (fixed at 0.5)",
    "test_metrics": {k: (None if v != v else v) for k, v in r.scalars().items()},
    "latency_ms_per_event": float(tt.loc[chosen, "latency_ms"]),
    "ensemble_weights_fallback": cfg.models.ensemble.model_dump(),
    "alert_config": cfg.alerts.model_dump(),
}
out = REPORT_DIR / "production_config.json"
out.write_text(json.dumps(production, indent=2, default=str))
print("wrote", out.relative_to(ROOT))
display(pd.Series(production["test_metrics"], name=chosen).to_frame().style.format("{:.3f}", na_rep="-"))

,FPR <= 1%,event recall >= 50%,latency <= 50 ms,all,weighted_f1,f1,pa_f1
model,,,,,,,
Stacked (LightGBM),True,True,True,True,0.551,0.644,0.664
LSTM-Cls,True,False,True,False,0.454,0.576,0.593
Weighted avg,True,False,True,False,0.421,0.562,0.582
Statistical,True,False,True,False,0.398,0.541,0.554
Isolation Forest,True,False,True,False,0.301,0.459,0.469
LSTM-AE,False,True,True,False,0.072,0.150,0.153


### Selected: **Stacked (LightGBM)** at threshold 0.500

wrote reports/production_config.json


,Stacked (LightGBM)
precision,0.930
recall,0.492
f1,0.644
macro_f1,0.515
weighted_f1,0.551
roc_auc,0.928
auc_pr,0.736
fpr,0.002
fnr,0.508
specificity,0.998



### Decision

**Run the stacked ensemble (LightGBM meta-learner over the four detectors) at decision threshold 0.5** - the setting recorded in `reports/production_config.json`. It is the only model that meets all three production constraints (FPR <= 1%, event recall >= 50%, latency <= 50 ms) and it ranks first on weighted F1, F1, PA F1, AUC-PR and AUC-ROC. Its cost is about 1 ms per event, which is well inside the streaming budget.

Deployment notes that follow from the evidence:

1. **Keep the statistical detector as the always-on fast path.** It is essentially perfect on point anomalies (F1 0.999, FPR 0) at 0.03 ms and needs no model artefacts, so it can alert immediately and keep working when a neural model is warming up or unavailable.
2. **Use the weighted-average fallback only as a degraded mode.** It loses about 8 F1 points and has three times the false-alarm rate of the stacker, so an alert when it is active should say so.
3. **Do not deploy the LSTM autoencoder on its own.** Its value is as a stacker input; fix its ranking quality (AUC-PR 0.078) before expecting more from it.
4. **Next priority is contextual anomalies** (60% of events, best recall 23%), for example through time-of-day/zone-conditioned baselines or longer context windows, followed by re-running this notebook with several seeds to put confidence intervals on the leaderboard.

Artefacts written by this notebook: `reports/<model>_<timestamp>.html` (one per model), `reports/leaderboard.{csv,html}`, `reports/leaderboard_default_threshold.{csv,html}`, `reports/production_config.json`.
